# Smoke test — `primekg` — one anonymized example
Validates the full pipeline on **primekg**: KG load → evidence dossier → drug-name anonymization → model call → JSON parse.

Run this notebook from inside the repo (it auto-finds `config.yaml`). Pick the model in the next cell; its key must be in `../.env`.

In [1]:
import os
# Model to smoke-test (key must be in .env). Swap as needed:
os.environ['BKG_MODELS'] = 'gemini:gemini-3.5-flash'   # or 'gemini:gemini-2.0-flash' / 'groq:llama-3.3-70b-versatile'
KG_TO_TEST = 'primekg'


In [2]:
# Bootstrap: exec the harness setup cells from 09 (paths, config, prompt, parse, dispatch, kg-block, queries)
import json, os
_p = os.path.join(os.path.dirname(os.getcwd()), 'eval_notebooks', '09_llm_integration.ipynb')
_p = _p if os.path.exists(_p) else 'eval_notebooks/09_llm_integration.ipynb'
_p = _p if os.path.exists(_p) else '09_llm_integration.ipynb'
_nb = json.load(open(_p))
for _i in [2, 4, 6, 8, 10, 12, 14, 16]:
    exec(''.join(_nb['cells'][_i]['source']), globals())
KGS = [KG_TO_TEST]   # restrict to the single KG under test
print('setup OK | model:', MODELS, '| KG:', KGS, '| anonymize:', ANONYMIZE, '| n_queries:', len(QUERIES))


repo root: /Users/shil6661/biokgsuite
MODE=real  models=['gemini:gemini-3.5-flash']
KGS=['primekg', 'drkg', 'biokg', 'openbilink']  conditions=['no_kg', 'kg']  n_diseases=22  pool=8  shuffles=2
22 queries  (pool=8):
  - [Cardiovascular] cardiovascular disease (DOID:1287)  true drug: semaglutide
  - [Dermatology] vitiligo (DOID:12306)  true drug: ruxolitinib
  - [Immunology] polymyalgia rheumatica (DOID:853)  true drug: tocilizumab
  - [Oncology] multiple myeloma (DOID:9538)  true drug: teclistamab
  - [Respiratory] chronic obstructive pulmonary disease (DOID:3083)  true drug: dupilumab
  - [Cardiovascular] congestive heart failure (DOID:6000)  true drug: dapagliflozin
  - [Immunology] alopecia areata (DOID:986)  true drug: baricitinib
  - [Oncology] hepatocellular carcinoma (DOID:684)  true drug: cabozantinib
  - [Immunology] hidradenitis suppurativa (DOID:0060480)  true drug: adalimumab
  - [Oncology] gastrointestinal neuroendocrine tumor (DOID:0080909)  true drug: everolimus
  - [Imm

In [3]:
# Build ONE anonymized example for this KG (prefer a disease the KG actually covers)
import numpy as np
block_for, disease_profile_fn = make_kg_block_fn(KG_TO_TEST, bridge_mode=BRIDGE_MODE, cap=CAP,
                                                 anonymize=ANONYMIZE, anonymize_genes=ANONYMIZE_GENES)
model = MODELS[0]
def _build(q):
    ids = sorted({c['drug_id'] for c in q['candidates']})
    cmap = {d: f'Drug-{i+1}' for i, d in enumerate(ids)} if ANONYMIZE else None
    rng = np.random.default_rng(SEED)
    view, pos = assign_letters_and_evidence(q['candidates'], block_for, q['disease_id'], 'kg', MOCK, rng, code_map=cmap)
    return view, pos
chosen = None
for q in QUERIES:
    view, pos = _build(q)
    if any(c['letter'] == pos and c.get('evidence') for c in view):
        chosen = (q, view, pos); break
if chosen is None:
    q = QUERIES[0]; view, pos = _build(q); chosen = (q, view, pos)
    print('(note: no KG evidence found for the true drug in any query — showing first query anyway)')
q, view, pos = chosen
dprof = disease_profile_fn(q['disease_id'], q['disease_name'])
prompt = build_prompt(q['disease_name'], q['disease_id'], view, disease_profile=dprof)
print('='*72)
print('KG          :', KG_TO_TEST)
print('DISEASE     :', q['disease_name'], '(', q['disease_id'], ')')
print('TRUE drug is hidden as:', pos, '  (drug names anonymized as Drug-1..8)')
print('='*72)
print(prompt)


  [primekg] mapped 27671 numeric gene IDs -> symbols
KG          : primekg
DISEASE     : cardiovascular disease ( DOID:1287 )
TRUE drug is hidden as: F   (drug names anonymized as Drug-1..8)
You are assisting a drug repurposing scientist. For the target disease below,
you are given candidate drugs. Some include evidence retrieved from a
biomedical knowledge graph, reported as mechanistic links
(drug -> target -> pathway -> disease) and related annotations.


Target disease: cardiovascular disease (DOID:1287)
Disease profile (from the knowledge graph): cardiovascular disease is related to cardiovascular neoplasm; cardiovascular disease is related to congenital anomaly of cardiovascular system; cardiovascular disease is related to autoimmune disease of cardiovascular system; cardiovascular disease is related to disease by anatomical system; cardiovascular disease is related to vascular disease; cardiovascular disease is related to heart disease

Candidates (listed order is arbitrary):
A.

In [4]:
# Single model call + parse
resp = rank_call(model, prompt, view, 0)
print('RAW RESPONSE  (', model, ')\n' + '-'*72)
print(resp)
print('-'*72)
is_err = isinstance(resp, str) and resp.startswith('__ERROR__')
ordered, fields = parse_response(resp, [c['letter'] for c in view])
print('parsed ranking :', ordered)
print('true-drug rank :', rank_of(pos, ordered, POOL_SIZE), '/', POOL_SIZE)
ok = (not is_err) and bool(ordered) and len(ordered) == len(view)
print('\nSMOKE TEST PASSED \u2705  (KG loaded, evidence built, anonymized, full ranking parsed)' if ok
      else '\nSMOKE TEST FAILED \u274c  — API error or incomplete parse; see raw response above')


RAW RESPONSE  ( gemini:gemini-3.5-flash )
------------------------------------------------------------------------
{
  "reasoning": "Drug-7 (F) is the most promising candidate because it targets GLP1R and MME, which are highly validated therapeutic targets for cardiovascular diseases (MACE reduction and heart failure management). Drug-1 (A) is also highly plausible as MTOR inhibitors are clinically used in drug-eluting stents for vascular disease. Drug-8 (H) targets GAA, which is relevant for the specific cardiological manifestation of Pompe disease. In contrast, JAK inhibitors (Drug-2/D and Drug-6/G) and BTK inhibitors (Drug-5/C) are associated with significant cardiovascular risks and black-box warnings, making them unfavorable for cardiovascular disease repurposing.",
  "ranking": [
    {
      "rank": 1,
      "drug": "F",
      "basis": "both",
      "evidence_agreement": "consistent",
      "confidence": 5,
      "rationale": "Targets GLP1R and MME (neprilysin), which are highly 